# 04 · Dokładni najbliżsi sąsiedzi i pełny retrieval

Przed przybliżonym indeksem zbudujemy referencyjny retriever: policzymy cosinus pytania z **każdym** fragmentem i posortujemy wszystkie wyniki. To jest wyszukiwanie semantyczne typu exact kNN. Ma koszt proporcjonalny do liczby wektorów i ich wymiaru; przy małym zbiorze daje prosty punkt odniesienia.

In [ ]:
# Jedyna komórka z importami i ścieżką projektu.
import hashlib
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, section_records

## 1. Cosinus na trzech małych wektorach

Wektor bliski kierunkowi pytania dostaje wysoki wynik niezależnie od skali. Zero-wymiarowy lub zerowy wektor wymaga osobnej obsługi; embedding modelu nie powinien go zwracać, ale kod produkcyjny musi to sprawdzić.

In [ ]:
toy_query = np.array([1., 0.])
toy_docs = np.array([[2., 0.], [1., 1.], [-1., 0.]])
toy_cosines = (toy_docs @ toy_query) / (np.linalg.norm(toy_docs, axis=1) * np.linalg.norm(toy_query))
toy_cosines.tolist()

## 2. Indeks dokładny w pamięci

Każda pozycja macierzy odpowiada dokładnie jednemu `record`. `argsort` zwraca pozycje malejąco. To jest pełny skan, bez ANN i bez generowania odpowiedzi. Do obliczeń normalizujemy oba końce, więc iloczyn skalarny = cosinus.

In [ ]:
records = section_records()
matrix = embed_texts([row["text"] for row in records])
assert np.all(np.linalg.norm(matrix, axis=1) > 0)
matrix = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
question = "Why is a percentage forecasting metric undefined with zero demand?"
q = embed_texts([question])[0]
q = q / np.linalg.norm(q)
scores = matrix @ q
top = np.argsort(-scores)[:5]
[{"rank": rank, "id": records[int(i)]["id"], "heading": records[int(i)]["heading"],
  "cosine": round(float(scores[i]), 3)} for rank, i in enumerate(top, 1)]

## 3. Kandydat ≠ dowód ≠ odpowiedź

Pobierz dokładny tekst po ID i sprawdź wersję/hash przed cytowaniem. Ranking może znaleźć bliski tematycznie, lecz niewystarczający fragment. Metadane `access_scope` muszą ograniczać kandydatów **przed** ujawnieniem tekstu. Tu pokazujemy fikcyjny korpus dydaktyczny, bez mechanizmu autoryzacji portalu Acme. Hash wykrywa zmianę tekstu; sam nie potwierdza prawdziwości treści.

In [ ]:
winner = records[int(top[0])]
assert winner["version"] == 1
assert (ROOT / winner["source_path"]).is_file()
assert hashlib.sha256(winner["text"].encode("utf-8")).hexdigest() == winner["content_hash"]
{"id": winner["id"], "source": winner["source_path"], "sha256": winner["content_hash"],
 "excerpt": winner["text"][:550]}

## 4. Mała ewaluacja retrievalu

Dla pytania przygotowaliśmy etykietę właściwego **dokumentu**, a nie tylko słowo kluczowe. `hit@k` odpowiada „czy w pierwszych k jest choć jeden jego fragment?”. To słabsza miara niż ręczna ocena wystarczalności, ale lepsza niż stwierdzenie „wynik wygląda dobrze”. Sprawdź dodatkowo pytanie o listopadowy billing i pytanie bez odpowiedzi w korpusie.

In [ ]:
checks = [
    ("Why is percentage error undefined at zero demand?", "SCI-FORECAST-2011"),
    ("What was November 2011 header billing?", "FIN-YTD-2011"),
    ("What is a receipt allocation?", "FIN-CASH-2011"),
]
evaluation = []
for text, expected_doc in checks:
    vector = embed_texts([text])[0]
    vector /= np.linalg.norm(vector)
    ranked = np.argsort(-(matrix @ vector))[:5]
    evaluation.append({"query": text, "expected": expected_doc,
                       "hit@1": records[int(ranked[0])]["document_id"] == expected_doc,
                       "hit@5": any(records[int(i)]["document_id"] == expected_doc for i in ranked)})
evaluation

Nawet pytanie bez odpowiedzi dostaje „najbliższy” fragment. Ta liczba jest rankingiem względnym, nie skalibrowanym prawdopodobieństwem poprawnej odpowiedzi. Próg odrzucenia trzeba dobrać na oznaczonych pytaniach, także takich bez odpowiedzi.

In [ ]:
unknown = "What was Acme's net profit after tax in 2011?"
unknown_vector = embed_texts([unknown])[0]
unknown_vector /= np.linalg.norm(unknown_vector)
unknown_rank = np.argsort(-(matrix @ unknown_vector))[:3]
[(records[int(i)]["id"], round(float(matrix[int(i)] @ unknown_vector), 3)) for i in unknown_rank]

### Dlaczego cosinus?

Cosinus jest prosty do interpretacji jako podobieństwo **kierunków** i pasuje do znormalizowanych wektorów. Nie jest uniwersalnie najlepszą metryką: wybór zależy od treningu modelu embeddingowego, rozkładu danych i ewaluacji. Przy normalizacji L2 ranking kosinusa i iloczynu skalarnego jest ten sam; odległość Euklidesowa na wektorach jednostkowych daje ten sam porządek, bo `||a-b||² = 2-2·cos(a,b)`.

**Ćwiczenie:** oznacz ręcznie kilka pytań, na które korpus nie odpowiada, i porównaj ich wyniki z pytaniami wspartymi źródłem. Czy jeden próg wystarcza dla obu grup?